# Scryfall Tags: Multi-Label Classification  
__Objective:__ Since the scryfall tags are in essence a collection of multiple labels for each card, this problem is at it's core a mutli-label classification task. However, given that there are nearly as many labels as cards, it will be easier to frame this as a seq2seq problem.

## Packages and Data

In [1]:
# # UNCOMMENT if operating in google colab

# ## mount to drive
# from google.colab import drive
# drive.mount('/content/gdrive', force_remount=True)

# ## ensure current directory is identified
# import os
# PROJECT_PATH = '/content/gdrive/MyDrive/data_science/scryfall-llm-sandbox'
# os.chdir(f"{PROJECT_PATH}/notebooks")
# print(f"Current Directory: {os.getcwd()}")

# # check that GPUs are available
# import torch
# print(f'GPUs Available?: {torch.cuda.is_available()}')
# print(f"Device Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

In [2]:
# packages

## connect project directory
import sys
from pathlib import Path
dir = str(Path(Path.cwd()).parents[0])
if dir not in sys.path:
    sys.path.append(dir)

# ## UNCOMMENT if operating in gdrive colab
# gdir = str(Path.cwd()) +  '/MyDrive/data_science/scryfall-llm-sandbox'
# if gdir not in sys.path:
#   sys.path.append(gdir)

## load from project directory
from src.data_gathering.scryfall_dataset import ScryfallDataset
from src.fine_tuning.modeling import FineTuneLLM

In [3]:
# params

## data gathering
from src.config import BUILD_DATASET, TASK, DATASET_SIZE_N, TEST_SIZE_N
from src.config import MAX_INPUT_LENGTH, MAX_TARGET_LENGTH

## modeling
from src.config import MODEL_NAME
from src.config import BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, NUM_EPOCHS
from src.config import GENERATION_MAX_LENGTH, GENERATION_NUM_BEAMS

## save model
from src.config import OUTPUT_DIR

In [4]:
# get data
sf = ScryfallDataset(task = TASK)

## build dataset as needed
if BUILD_DATASET:
    sf.build_dataset(
        tag_path = '../reports/scryfall_tags.json',
        train_size_pct = 0.8,
        truncate_dataset = DATASET_SIZE_N,
        test_size_n = TEST_SIZE_N
    )

## load dataset
sf.load_hf_dataset(
    train_path = f'../data/scryfall_{TASK}_train.json',
    val_path = f'../data/scryfall_{TASK}_val.json',
    test_path = f'../data/scryfall_{TASK}_test.json'
)

Scryfall Tag Question Answering Dataset Built
	Train Records = 692
	Validation Records = 174
	Test Records = 10
	Records saved to...
		../data/scryfall_seq2seq_train.json
		../data/scryfall_seq2seq_val.json
		../data/scryfall_seq2seq_test.json
	NOTE: This method does not create the huggingface dataset object. Run load_dataset() for that.
Scryfall Tag Seq2Seq Dataset Loaded
	Train Records = 692
	Val Records = 174
	Test Records = 10
	Count Unique Tags = 0


## Modeling

In [5]:
# from transformers import Trainer
# from torch.nn import BCEWithLogitsLoss

# class CustomTrainer(Trainer):
#     def __init__(self, pos_weights, *args, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.pos_weights = pos_weights

#     def compute_loss(self, model, inputs, return_outputs = False):
#         labels = inputs.pop("labels")
#         outputs = model(**inputs)
#         logits = outputs.logits

#         loss_fct = BCEWithLogitsLoss(pos_weight = self.pos_weights)
#         loss = loss_fct(logits, labels)

#         return (loss, outputs) if return_outputs else loss

In [ ]:
# fine tune the model
tagger = FineTuneLLM(
    model_name = MODEL_NAME,
    dataset = sf.dataset
)
tagger.prepare_data(
    max_input_length = MAX_INPUT_LENGTH,
    max_target_length = MAX_TARGET_LENGTH
)
tagger.train(
    batch_size = BATCH_SIZE,
    n_epochs = NUM_EPOCHS,
    learning_rate = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY,
    generation_max_length = GENERATION_MAX_LENGTH,
    generation_num_beams = GENERATION_NUM_BEAMS,
    # output_dir = f'{PROJECT_PATH}/models/scryfall_auto_tagger' # uncomment if in google colab
    output_dir = f'../models/scryfall_auto_tagger'
)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 344,064 || all params: 77,305,216 || trainable%: 0.4451


Map:   0%|          | 0/692 [00:00<?, ? examples/s]

Map:   0%|          | 0/174 [00:00<?, ? examples/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'pad_token_id': 0}.


Using device: NVIDIA GeForce GTX 1650 with Max-Q Design


c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\transformers\data\data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\bld\libtorch_1770197074191\work\torch\csrc\utils\tensor_new.cpp:256.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Epoch,Training Loss,Validation Loss,Micro Precision,Micro Recall,Micro F1
1,501.457581,472.992310,0,0.000000,0
2,404.655121,356.204529,0,0.000000,0
3,322.429535,291.107452,0,0.000000,0
4,294.217316,260.549408,0,0.000000,0
5,264.765106,251.901764,0,0.000000,0


c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\peft\utils\save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\peft\utils\save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\peft\utils\save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\peft\utils\save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\peft\utils\save_and

hilda crown of winter

In [7]:
# Usage:
# Assuming 'tagger' is your FineTuneLLM instance
from src.utils.debug_autotagger import debug_autotagger_outputs
debug_autotagger_outputs(tagger, sf.dataset, num_samples=10)


--- Model Output Debugging (10 Samples) ---
[GT = Ground Truth, PR = Model Prediction]

Sample 1:
  GT: tags: <tag> cycle-da1-charm </tag> <tag> discard </tag> <tag> tutor-instant </tag> <tag> rummage </tag> <tag> counterspell-tuck </tag> <tag> charm </tag> <tag> tutor-to-hand </tag> <tag> synergy-blue </tag>
  PR: [EMPTY]

Sample 2:
  GT: tags: <tag> intervening if clause </tag> <tag> life for cards </tag> <tag> alliteration </tag> <tag> sacrifice outlet-creature </tag> <tag> repeatable pure draw </tag> <tag> evasion </tag> <tag> activated ability </tag> <tag> repeatable lifegain </tag> <tag> cards in graveyard matter </tag> <tag> keyword errata surveil </tag> <tag> draw engine </tag> <tag> surveil </tag>
  PR: [EMPTY]

Sample 3:
  GT: tags: <tag> punisher </tag> <tag> landfall </tag> <tag> repeatable pure draw </tag> <tag> repeatable seek </tag> <tag> evasion </tag> <tag> tap fuel-land </tag> <tag> tutor-land-to-battlefield </tag> <tag> gives tap ability </tag> <tag> intervening if 

In [8]:
record = sf.dataset['test'][5]
pred_tags = tagger.generate_tags(card_text = record['document'])
print(f'Card\n{record["document"]}')
print(f'Actual Tags = {record["tags"]}')
print(f'Predicted Tags = {pred_tags}')

Card

        mtg card scryfall tags task:
        Return tags for the following card using this format:
        <tag> example tag </tag>

        ----------
        Card = Blind Creeper
        Mana Cost = {1}{B}
Mana Value = 2.0

        Type Line = Creature — Zombie Beast

        Rules Text = Whenever a player casts a spell, this creature gets -1/-1 until end of turn.
 
        Power = 3
Toughness = 3

        
        Color Identity = ['B']

        Rarity = common
        ----------
        
Actual Tags = tags: <tag> drawback </tag> <tag> cast trigger </tag>
Predicted Tags = set()


## Save To Huggingface Hub

In [9]:
# # UNCOMMENT TO login to the hugging face
# from huggingface_hub import notebook_login
# # with open('../huggingface_token.txt', 'r') as f:
# #     token = f.read()

# notebook_login()

In [10]:
# # UNCOMMENT TO upload to huggingface hub
# from huggingface_hub import notebook_login
# from huggingface_hub import get_full_repo_name
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# repo_id = get_full_repo_name(OUTPUT_DIR)
# tagger.model.push_to_hub(repo_id)
# tagger.tokenizer.push_to_hub(repo_id)

## Graveyard

In [11]:
# # upload the model to the huggingface hub
# from huggingface_hub import Repository
# from huggingface_hub import get_full_repo_name

# ## define the repo locally
# ## NOTE: Be sure to create OUTPUT_DIR in the hub manually first
# repo_name = get_full_repo_name(OUTPUT_DIR)
# repo = Repository(OUTPUT_DIR, clone_from = repo_name)

# ## save to hub
# tagger.save_to_huggingface_hub(
#     output_dir = OUTPUT_DIR,
#     repo = repo,
#     commit_message = f'Fine-tuned {MODEL_NAME} on scryfall tags.'
# )